# Atividade 2 — Data Lake com S3 e Athena (venda de ingressos)

Roteiro de execução do pipeline completo: ingestão, Data Quality, quarentena,
camadas Silver e Gold, e auditoria no Athena. Rodado direto no Google Colab.


### 1. Instalando as bibliotecas

In [12]:
# boto3 pra falar com a AWS, pandas pra manipular os dados, pyarrow pra gravar parquet
!pip install -q boto3 pandas pyarrow


### 2. Login na AWS
Colei minhas credenciais aqui com `getpass` pra não ficar nada salvo no notebook.

In [13]:
import getpass
import boto3

# uso getpass pra digitar sem aparecer na tela e sem salvar no notebook
AWS_ACCESS_KEY_ID = getpass.getpass("AWS_ACCESS_KEY_ID: ")
AWS_SECRET_ACCESS_KEY = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
AWS_SESSION_TOKEN = getpass.getpass("AWS_SESSION_TOKEN (Enter se não tiver): ") or None
AWS_REGION = input("AWS_REGION (ex.: us-east-1): ") or "us-east-1"

session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION,
)

s3 = session.client("s3")
athena = session.client("athena")

print("Conectado na região:", AWS_REGION)


AWS_ACCESS_KEY_ID: ··········
AWS_SECRET_ACCESS_KEY: ··········
AWS_SESSION_TOKEN (Enter se não tiver): ··········
AWS_REGION (ex.: us-east-1): us-east-1
Conectado na região: us-east-1


### 3. Configurações do projeto
Nome do bucket, banco do Athena e a partição do dia.

In [14]:
import datetime

BUCKET_NAME = "datalake-amanda-10781761"          # troquei pelo meu bucket antes de rodar
DATABASE_NAME = "datalake_db_" + BUCKET_NAME.replace("-", "_")
INGEST_DATE = datetime.date.today().isoformat()   # partição do dia

QTD_COMPRADORES = 500
QTD_EVENTOS = 50
QTD_VENDAS = 2000

print("Bucket:", BUCKET_NAME)
print("Banco Athena:", DATABASE_NAME)
print("Partição de hoje:", INGEST_DATE)


Bucket: datalake-amanda-10781761
Banco Athena: datalake_db_datalake_amanda_10781761
Partição de hoje: 2026-09-13


### 4. Criando o bucket e a estrutura de pastas

In [15]:
def criar_bucket(bucket_name, region):
    try:
        if region == "us-east-1":
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region},
            )
        print(f"[OK] Bucket '{bucket_name}' criado.")
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"[OK] Bucket '{bucket_name}' já existia, segue o jogo.")
    except Exception as e:
        print(f"[AVISO] {e}")

criar_bucket(BUCKET_NAME, AWS_REGION)

# só carimbando as pastas principais pra ficar visível no console do S3
pastas = [
    "raw/compradores/", "raw/eventos/", "raw/vendas_ingressos/",
    "quarantine/vendas_rejeitadas/",
    "processed/fato_vendas_ingressos/",
    "gold/vendas_uf_categoria/",
    "athena-results/",
]
for pasta in pastas:
    s3.put_object(Bucket=BUCKET_NAME, Key=pasta)
print("[OK] Estrutura de pastas criada.")


[OK] Bucket 'datalake-amanda-10781761' criado.
[OK] Estrutura de pastas criada.


### 5. Gerando os dados (com anomalias de propósito)
Criei `compradores`, `eventos` e `vendas_ingressos`. Nas vendas, forcei 3 tipos de erro (5% de chance cada): quantidade inválida, comprador que não existe e evento que não existe.

In [16]:
import random
import pandas as pd

random.seed(42)

nomes = ["Andre", "Debora", "Rafael", "Felipe", "Tatiane", "Juliana", "Danilo",
         "Flavia", "Vinicius", "Beatriz", "Rodrigo", "Bruna", "Alexandre",
         "Renata", "Caio", "Kelly", "Marcos", "Camila", "Gabriel", "Mariana"]
sobrenomes = ["Silva", "Santos", "Oliveira", "Souza", "Pereira", "Costa",
              "Rodrigues", "Almeida", "Nascimento", "Lima"]
cidades = [("São Paulo", "SP"), ("Campinas", "SP"), ("Rio de Janeiro", "RJ"),
           ("Belo Horizonte", "MG"), ("Curitiba", "PR"), ("Florianópolis", "SC"),
           ("Porto Alegre", "RS"), ("Salvador", "BA"), ("Recife", "PE"),
           ("Goiânia", "GO"), ("Brasília", "DF")]

# nome do evento, categoria, faixa de preço do ingresso
catalogo_eventos = [
    ("Festival de Verão", "Show", 80, 450), ("Turnê Nacional", "Show", 120, 800),
    ("Clássico Estadual", "Esporte", 40, 350), ("Final de Campeonato", "Esporte", 100, 900),
    ("Comédia Stand-up", "Teatro", 50, 250), ("Musical da Broadway", "Teatro", 150, 600),
    ("Estreia de Blockbuster", "Cinema", 20, 60), ("Sessão Cult", "Cinema", 15, 40),
    ("Feira Gastronômica", "Festival", 30, 120), ("Festival de Cerveja Artesanal", "Festival", 40, 180),
    ("Convenção de Games", "Convenção", 60, 300), ("Expo de Quadrinhos", "Convenção", 50, 250),
]

# --- compradores ---
compradores = pd.DataFrame([{
    "comprador_id": i + 1,
    "nome": f"{random.choice(nomes)} {random.choice(sobrenomes)}",
    "cidade": (c := random.choice(cidades))[0],
    "estado": c[1],
} for i in range(QTD_COMPRADORES)])

# --- eventos ---
eventos = pd.DataFrame([{
    "evento_id": i + 1,
    "nome": (e := random.choice(catalogo_eventos))[0],
    "categoria": e[1],
    "preco_ingresso": round(random.uniform(e[2], e[3]), 2),
} for i in range(QTD_EVENTOS)])

# --- vendas de ingressos, já com as anomalias ---
PCT_QTD_INVALIDA = 0.05
PCT_COMPRADOR_INEXISTENTE = 0.05
PCT_EVENTO_INEXISTENTE = 0.05

data_inicial = datetime.date(2026, 1, 1)
linhas = []
contadores = {"validos": 0, "quantidade_invalida": 0, "comprador_inexistente": 0, "evento_inexistente": 0}

for i in range(QTD_VENDAS):
    comprador_id = random.randint(1, QTD_COMPRADORES)
    evento_id = random.randint(1, QTD_EVENTOS)
    quantidade = random.randint(1, 6)   # ingressos comprados de uma vez

    sorteio = random.random()
    if sorteio < PCT_QTD_INVALIDA:
        quantidade = random.choice([0, -1, -2, -5])   # força a anomalia
        contadores["quantidade_invalida"] += 1
    elif sorteio < PCT_QTD_INVALIDA + PCT_COMPRADOR_INEXISTENTE:
        comprador_id = QTD_COMPRADORES + random.randint(1000, 9999)   # id que não existe
        contadores["comprador_inexistente"] += 1
    elif sorteio < PCT_QTD_INVALIDA + PCT_COMPRADOR_INEXISTENTE + PCT_EVENTO_INEXISTENTE:
        evento_id = QTD_EVENTOS + random.randint(1000, 9999)   # id que não existe
        contadores["evento_inexistente"] += 1
    else:
        contadores["validos"] += 1

    linhas.append({
        "venda_id": i + 1,
        "comprador_id": comprador_id,
        "evento_id": evento_id,
        "quantidade": quantidade,
        "data_venda": (data_inicial + datetime.timedelta(days=random.randint(0, 364))).isoformat(),
    })

vendas_ingressos = pd.DataFrame(linhas)

print("Compradores:", len(compradores), "| Eventos:", len(eventos), "| Vendas:", len(vendas_ingressos))
print(contadores)


Compradores: 500 | Eventos: 50 | Vendas: 2000
{'validos': 1694, 'quantidade_invalida': 96, 'comprador_inexistente': 104, 'evento_inexistente': 106}


### 6. Subindo pra camada Raw
Cada CSV vai pro S3 particionado por `ingest_date`.

In [17]:
def upload_csv(df, nome_tabela):
    caminho_local = f"/tmp/{nome_tabela}.csv"
    df.to_csv(caminho_local, index=False, encoding="utf-8")
    key = f"raw/{nome_tabela}/ingest_date={INGEST_DATE}/{nome_tabela}.csv"
    s3.upload_file(caminho_local, BUCKET_NAME, key)
    print(f"[OK] {nome_tabela} -> s3://{BUCKET_NAME}/{key}")

upload_csv(compradores, "compradores")
upload_csv(eventos, "eventos")
upload_csv(vendas_ingressos, "vendas_ingressos")


[OK] compradores -> s3://datalake-amanda-10781761/raw/compradores/ingest_date=2026-09-13/compradores.csv
[OK] eventos -> s3://datalake-amanda-10781761/raw/eventos/ingest_date=2026-09-13/eventos.csv
[OK] vendas_ingressos -> s3://datalake-amanda-10781761/raw/vendas_ingressos/ingest_date=2026-09-13/vendas_ingressos.csv


### 7. Data Quality, quarentena e camada Silver
Descarto venda com quantidade ≤ 0, comprador que não existe ou evento que não existe. O que sobrou de errado vai pra quarentena em JSON, com o motivo. O que ficou certo eu enriqueço e calculo o valor_total.

In [18]:
import pyarrow  # só pra garantir que o engine parquet tá disponível

compradores_ids = set(compradores["comprador_id"])
eventos_ids = set(eventos["evento_id"])

def motivo_rejeicao(row):
    # confere as 3 regras e junta os motivos que bateram
    motivos = []
    if row["quantidade"] <= 0:
        motivos.append("quantidade_invalida")
    if row["comprador_id"] not in compradores_ids:
        motivos.append("comprador_inexistente")
    if row["evento_id"] not in eventos_ids:
        motivos.append("evento_inexistente")
    return motivos

vendas_ingressos["motivos_rejeicao"] = vendas_ingressos.apply(motivo_rejeicao, axis=1)
vendas_ingressos["valido"] = vendas_ingressos["motivos_rejeicao"].apply(len) == 0

vendas_validas = vendas_ingressos[vendas_ingressos["valido"]].drop(columns=["valido", "motivos_rejeicao"])
vendas_invalidas = vendas_ingressos[~vendas_ingressos["valido"]].drop(columns=["valido"])

print(f"Válidas: {len(vendas_validas)} | Rejeitadas: {len(vendas_invalidas)}")

# --- quarentena: um JSON por linha ---
quarentena_local = "/tmp/vendas_rejeitadas.json"
vendas_invalidas.to_json(quarentena_local, orient="records", lines=True, force_ascii=False)
quarentena_key = f"quarantine/vendas_rejeitadas/data={INGEST_DATE}/rejeitados.json"
s3.upload_file(quarentena_local, BUCKET_NAME, quarentena_key)
print(f"[OK] Quarentena -> s3://{BUCKET_NAME}/{quarentena_key}")

# --- camada Silver: junta tudo e calcula o valor_total ---
fato_vendas = (
    vendas_validas
    .merge(compradores, on="comprador_id", how="left")
    .merge(eventos, on="evento_id", how="left")
)
fato_vendas["valor_total"] = fato_vendas["quantidade"] * fato_vendas["preco_ingresso"]

silver_local = "/tmp/fato_vendas_ingressos.parquet"
fato_vendas.to_parquet(silver_local, index=False)
silver_key = f"processed/fato_vendas_ingressos/ingest_date={INGEST_DATE}/fato_vendas_ingressos.parquet"
s3.upload_file(silver_local, BUCKET_NAME, silver_key)
print(f"[OK] Silver -> s3://{BUCKET_NAME}/{silver_key}")

fato_vendas.head()


Válidas: 1694 | Rejeitadas: 306
[OK] Quarentena -> s3://datalake-amanda-10781761/quarantine/vendas_rejeitadas/data=2026-09-13/rejeitados.json
[OK] Silver -> s3://datalake-amanda-10781761/processed/fato_vendas_ingressos/ingest_date=2026-09-13/fato_vendas_ingressos.parquet


,venda_id,comprador_id,evento_id,quantidade,data_venda,nome_x,cidade,estado,nome_y,categoria,preco_ingresso,valor_total
0,1,165,39,4,2026-02-01,Gabriel Pereira,Porto Alegre,RS,Sessão Cult,Cinema,30.78,123.12
1,2,231,7,3,2026-02-13,Alexandre Rodrigues,Belo Horizonte,MG,Expo de Quadrinhos,Convenção,58.46,175.38
2,4,269,34,5,2026-07-10,Felipe Souza,Curitiba,PR,Sessão Cult,Cinema,32.03,160.15
3,5,470,19,4,2026-06-23,Felipe Lima,Recife,PE,Festival de Cerveja Artesanal,Festival,96.06,384.24
4,6,348,39,1,2026-11-28,Danilo Souza,Goiânia,GO,Sessão Cult,Cinema,30.78,30.78


### 8. Camada Gold
Agrupei por estado e categoria e calculei as métricas de negócio.

In [19]:
gold = (
    fato_vendas
    .groupby(["estado", "categoria"], as_index=False)
    .agg(
        total_vendas=("venda_id", "count"),
        ingressos_total=("quantidade", "sum"),
        valor_total_vendido=("valor_total", "sum"),
    )
)
gold["ticket_medio"] = gold["valor_total_vendido"] / gold["total_vendas"]

gold_local = "/tmp/vendas_uf_categoria.parquet"
gold.to_parquet(gold_local, index=False)
gold_key = f"gold/vendas_uf_categoria/ingest_date={INGEST_DATE}/vendas_uf_categoria.parquet"
s3.upload_file(gold_local, BUCKET_NAME, gold_key)
print(f"[OK] Gold -> s3://{BUCKET_NAME}/{gold_key}")

gold.sort_values("valor_total_vendido", ascending=False).head(10)


[OK] Gold -> s3://datalake-amanda-10781761/gold/vendas_uf_categoria/ingest_date=2026-09-13/vendas_uf_categoria.parquet


,estado,categoria,total_vendas,ingressos_total,valor_total_vendido,ticket_medio
4,BA,Show,37,146,58625.03,1584.460270
58,SP,Show,47,161,53696.70,1142.482979
59,SP,Teatro,62,210,41061.30,662.279032
10,DF,Show,23,102,39390.36,1712.624348
16,GO,Show,25,89,33603.30,1344.132000
22,MG,Show,29,95,32948.60,1136.158621
52,SC,Show,24,95,31668.64,1319.526667
28,PE,Show,23,80,30796.42,1338.974783
55,SP,Convenção,54,180,29345.06,543.427037
35,PR,Teatro,29,128,27662.64,953.884138


### 9. Criando banco e tabelas no Athena
Crio as 6 tabelas externas e rodo o `MSCK REPAIR TABLE` pra registrar as partições.

In [20]:
import time

ATHENA_OUTPUT = f"s3://{BUCKET_NAME}/athena-results/"

def run_query(sql, database=None, wait=True):
    # dispara a query e (se wait=True) fica checando até terminar
    kwargs = {
        "QueryString": sql,
        "ResultConfiguration": {"OutputLocation": ATHENA_OUTPUT},
    }
    if database:
        kwargs["QueryExecutionContext"] = {"Database": database}
    exec_id = athena.start_query_execution(**kwargs)["QueryExecutionId"]

    if not wait:
        return exec_id

    while True:
        status = athena.get_query_execution(QueryExecutionId=exec_id)["QueryExecution"]["Status"]["State"]
        if status in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1.5)

    if status != "SUCCEEDED":
        reason = athena.get_query_execution(QueryExecutionId=exec_id)["QueryExecution"]["Status"].get("StateChangeReason", "")
        raise RuntimeError(f"Query {status}: {reason}\nSQL: {sql}")

    return exec_id

def query_to_dataframe(exec_id):
    # transforma o resultado bruto do Athena num DataFrame só pra ficar fácil de ler
    resultado = athena.get_query_results(QueryExecutionId=exec_id)
    colunas = [c["Label"] for c in resultado["ResultSet"]["ResultSetMetadata"]["ColumnInfo"]]
    linhas = []
    for row in resultado["ResultSet"]["Rows"][1:]:  # a primeira linha é o cabeçalho
        linhas.append([campo.get("VarCharValue") for campo in row["Data"]])
    return pd.DataFrame(linhas, columns=colunas)

# --- banco ---
run_query(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")
print(f"[OK] Banco '{DATABASE_NAME}' pronto.")

ddl_raw_compradores = f'''
CREATE EXTERNAL TABLE IF NOT EXISTS raw_compradores (
  comprador_id INT, nome STRING, cidade STRING, estado STRING
)
PARTITIONED BY (ingest_date STRING)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
LOCATION 's3://{BUCKET_NAME}/raw/compradores/'
TBLPROPERTIES ('skip.header.line.count'='1')
'''

ddl_raw_eventos = f'''
CREATE EXTERNAL TABLE IF NOT EXISTS raw_eventos (
  evento_id INT, nome STRING, categoria STRING, preco_ingresso DOUBLE
)
PARTITIONED BY (ingest_date STRING)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
LOCATION 's3://{BUCKET_NAME}/raw/eventos/'
TBLPROPERTIES ('skip.header.line.count'='1')
'''

ddl_raw_vendas = f'''
CREATE EXTERNAL TABLE IF NOT EXISTS raw_vendas_ingressos (
  venda_id INT, comprador_id INT, evento_id INT, quantidade INT, data_venda STRING
)
PARTITIONED BY (ingest_date STRING)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
LOCATION 's3://{BUCKET_NAME}/raw/vendas_ingressos/'
TBLPROPERTIES ('skip.header.line.count'='1')
'''

ddl_quarentena = f'''
CREATE EXTERNAL TABLE IF NOT EXISTS quarentena_vendas (
  venda_id INT, comprador_id INT, evento_id INT, quantidade INT,
  data_venda STRING, motivos_rejeicao ARRAY<STRING>
)
PARTITIONED BY (data STRING)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION 's3://{BUCKET_NAME}/quarantine/vendas_rejeitadas/'
'''

ddl_silver = f'''
CREATE EXTERNAL TABLE IF NOT EXISTS silver_fato_vendas_ingressos (
  venda_id INT, comprador_id INT, evento_id INT, quantidade INT, data_venda STRING,
  nome_comprador STRING, cidade STRING, estado STRING,
  nome_evento STRING, categoria STRING, preco_ingresso DOUBLE, valor_total DOUBLE
)
PARTITIONED BY (ingest_date STRING)
STORED AS PARQUET
LOCATION 's3://{BUCKET_NAME}/processed/fato_vendas_ingressos/'
'''

ddl_gold = f'''
CREATE EXTERNAL TABLE IF NOT EXISTS gold_vendas_uf_categoria (
  estado STRING, categoria STRING, total_vendas BIGINT,
  ingressos_total BIGINT, valor_total_vendido DOUBLE, ticket_medio DOUBLE
)
PARTITIONED BY (ingest_date STRING)
STORED AS PARQUET
LOCATION 's3://{BUCKET_NAME}/gold/vendas_uf_categoria/'
'''

for nome, ddl in [
    ("raw_compradores", ddl_raw_compradores),
    ("raw_eventos", ddl_raw_eventos),
    ("raw_vendas_ingressos", ddl_raw_vendas),
    ("quarentena_vendas", ddl_quarentena),
    ("silver_fato_vendas_ingressos", ddl_silver),
    ("gold_vendas_uf_categoria", ddl_gold),
]:
    run_query(ddl, database=DATABASE_NAME)
    print(f"[OK] Tabela '{nome}' criada.")

# registra as partições -- como já é Hive-style, o MSCK acha sozinho
for tabela in ["raw_compradores", "raw_eventos", "raw_vendas_ingressos", "quarentena_vendas",
               "silver_fato_vendas_ingressos", "gold_vendas_uf_categoria"]:
    run_query(f"MSCK REPAIR TABLE {tabela}", database=DATABASE_NAME)
    print(f"[OK] Partições registradas em '{tabela}'.")


[OK] Banco 'datalake_db_datalake_amanda_10781761' pronto.
[OK] Tabela 'raw_compradores' criada.
[OK] Tabela 'raw_eventos' criada.
[OK] Tabela 'raw_vendas_ingressos' criada.
[OK] Tabela 'quarentena_vendas' criada.
[OK] Tabela 'silver_fato_vendas_ingressos' criada.
[OK] Tabela 'gold_vendas_uf_categoria' criada.
[OK] Partições registradas em 'raw_compradores'.
[OK] Partições registradas em 'raw_eventos'.
[OK] Partições registradas em 'raw_vendas_ingressos'.
[OK] Partições registradas em 'quarentena_vendas'.
[OK] Partições registradas em 'silver_fato_vendas_ingressos'.
[OK] Partições registradas em 'gold_vendas_uf_categoria'.


### 10. Auditoria 1 — metadados ($path e $file_size)
Essa é uma das evidências.


In [21]:
sql_metadados = '''
SELECT
    "$path" AS arquivo,
    "$file_size" AS tamanho_bytes
FROM raw_vendas_ingressos
LIMIT 10
'''
exec_id = run_query(sql_metadados, database=DATABASE_NAME)
df_metadados = query_to_dataframe(exec_id)
df_metadados


,arquivo,tamanho_bytes
0,s3://datalake-amanda-10781761/raw/vendas_ingre...,48579
1,s3://datalake-amanda-10781761/raw/vendas_ingre...,48579
2,s3://datalake-amanda-10781761/raw/vendas_ingre...,48579
3,s3://datalake-amanda-10781761/raw/vendas_ingre...,48579
4,s3://datalake-amanda-10781761/raw/vendas_ingre...,48579
5,s3://datalake-amanda-10781761/raw/vendas_ingre...,48579
6,s3://datalake-amanda-10781761/raw/vendas_ingre...,48579
7,s3://datalake-amanda-10781761/raw/vendas_ingre...,48579
8,s3://datalake-amanda-10781761/raw/vendas_ingre...,48579
9,s3://datalake-amanda-10781761/raw/vendas_ingre...,48579


### 11. Auditoria 2 — conciliação (Raw = Silver + Quarentena)
Segunda evidência: confere se bate a soma.

In [22]:
sql_conciliacao = '''
WITH raw_count AS (
    SELECT COUNT(*) AS total_raw FROM raw_vendas_ingressos
),
silver_count AS (
    SELECT COUNT(*) AS total_silver FROM silver_fato_vendas_ingressos
),
quarentena_count AS (
    SELECT COUNT(*) AS total_quarentena FROM quarentena_vendas
)
SELECT
    raw_count.total_raw,
    silver_count.total_silver,
    quarentena_count.total_quarentena,
    (silver_count.total_silver + quarentena_count.total_quarentena) AS soma_silver_quarentena,
    CASE
        WHEN raw_count.total_raw = (silver_count.total_silver + quarentena_count.total_quarentena)
        THEN 'OK' ELSE 'DIVERGENTE'
    END AS integridade
FROM raw_count, silver_count, quarentena_count
'''
exec_id = run_query(sql_conciliacao, database=DATABASE_NAME)
df_conciliacao = query_to_dataframe(exec_id)
df_conciliacao


,total_raw,total_silver,total_quarentena,soma_silver_quarentena,integridade
0,2000,1694,306,2000,OK
